In [ ]:
import os, subprocess, glob
def sh(cmd):
    print('$', cmd, flush=True); r = subprocess.run(cmd, shell=True, executable='/bin/bash'); print('exit', r.returncode, flush=True); return r.returncode
sh('nvidia-smi | head -12; free -g | head -2')
sh('cd /tmp && git clone -q --depth 1 https://github.com/MolecularAI/REINVENT4 && ls REINVENT4 && ls REINVENT4/priors | head -30')
sh('pip install -q rdkit tensorboard molvs xxhash pydantic python-dotenv matplotlib "apted>=1.0.3,<2" "mmpdb>=2.1,<3" polars pillow pyyaml requests tqdm pytest pytest-mock 2>&1 | tail -3')
sh('cd /tmp/REINVENT4 && pip install -q --no-deps . 2>&1 | tail -2')
sh('mkdir -p /tmp/priors && curl -sL -o /tmp/priors/m2m_similarity.prior "https://zenodo.org/records/20701824/files/pubchem_ecfp4_with_count_with_rank_reinvent4_dict_voc.prior?download=1" && ls -la /tmp/priors')
import time, re
T0 = time.time()
par = glob.glob('/kaggle/input/**/parents.smi', recursive=True)[0]; os.makedirs('/kaggle/working/out', exist_ok=True)
from rdkit import Chem
OK = {6, 7, 8, 9, 16, 17, 35, 53}
good = []
for s in open(par).read().split():
    m = Chem.MolFromSmiles(s)
    if m is None or any(a.GetIsotope() for a in m.GetAtoms()) or any(a.GetAtomicNum() not in OK for a in m.GetAtoms()) or any(a.GetFormalCharge() not in (0, 1, -1) for a in m.GetAtoms()):
        continue
    good.append(Chem.MolToSmiles(m, isomericSmiles=False))
good = list(dict.fromkeys(good)); print('parents usable:', len(good), flush=True)
def toml(smis, n_smi, out):
    NL = chr(10); open('/tmp/p.smi', 'w').write(NL.join(smis) + NL)
    cfg = ['run_type = "sampling"', 'device = "cuda:0"', 'json_out_config = "/tmp/_s.json"', '', '[parameters]', 'model_file = "/tmp/priors/m2m_similarity.prior"', 'smiles_file = "/tmp/p.smi"',
           'sample_strategy = "multinomial"', 'temperature = 1.0', 'num_smiles = %d' % n_smi, 'unique_molecules = true', 'randomize_smiles = false', 'output_file = "%s"' % out]
    open('/tmp/s.toml', 'w').write(NL.join(cfg) + NL)
CH = 450
for k in range(0, len(good), CH):
    if time.time() - T0 > 50 * 60:
        print('time guard: stopping before chunk', k, flush=True); break
    toml(good[k:k + CH], 25, '/kaggle/working/out/mol2mol_samples_%d.csv' % (k // CH))
    sh('cd /tmp/REINVENT4 && reinvent -l /kaggle/working/out/chunk_%d.log /tmp/s.toml' % (k // CH))
sh('ls -la /kaggle/working/out')
print('NOTEBOOK_COMPLETED_OK')
